Auteur : Laetitia Ikusawa

Date : Mars 2026

Objectif : Update notebook P8_Notebook_Linux_EMR_PySpark_V1.0.ipynb to add broadcast Tensorflow and incude PCA
***

<details>
  <summary>Summary</summary>

  [Configuration](#configuration)
  
</details>



## Configuration


### Dependencies import

In [1]:
# ============================================================
# LOGGING AND WARNINGS
# ============================================================

import os
import warnings
import logging

# No tensorflow warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0=all, 1=info, 2=warning, 3=error only

# No warnings
warnings.filterwarnings("ignore")

# Set logging level for PySpark/Py4J
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)


In [2]:
import sys

print(f"Python version: {sys.version}")

try:
    import pyspark

    print(f"PySpark version: {pyspark.__version__}")
except ImportError:
    print("⚠️ PySpark not installed")

try:
    import tensorflow as tf

    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("⚠️ TensorFlow not installed")

try:
    from PIL import Image

    print(f"PIL/Pillow: OK")
except ImportError:
    print("⚠️ Pillow not installed")

Python version: 3.11.15 (main, Mar 16 2026, 23:13:10) [GCC 12.2.0]
PySpark version: 3.5.3
TensorFlow version: 2.21.0
PIL/Pillow: OK


In [2]:
# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, element_at, split, PandasUDFType
from pyspark.sql.types import ArrayType, FloatType, StructType, StructField, StringType
from pyspark.ml.feature import PCA, VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT


# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model

# Autres imports
from PIL import Image
import pandas as pd
import numpy as np
import io
import os
from pathlib import Path

print("✅ Imported")

✅ Imported


### Path definitions

In [3]:
# Automaticaly find project root
def find_project_root():
    """
    Find project root by searching for markers (.git, poetry.lock).
    Works regardless of where the Jupyter kernel is started.
    """
    # Try using __file__ if available (scripts .py)
    current = Path(__file__).parent if "__file__" in globals() else Path.cwd()

    # Search for root (contains .git or poetry.lock)
    for parent in [current] + list(current.parents):
        if (parent / ".git").exists() or (parent / "poetry.lock").exists():
            return parent

    # Fallback: if we are in notebooks/, go up one level
    if current.name == "notebooks":
        return current.parent
    elif (current / "notebooks").exists():
        return current

    # Last resort
    return current.parent


# Project paths
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
FEATURES_DIR = DATA_DIR / "features"
PCA_DIR = DATA_DIR / "pca"

# Create directories if they don't exist
for directory in [DATA_DIR, RAW_DATA_DIR, FEATURES_DIR, PCA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"📁 Project directory: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Raw data directory: {RAW_DATA_DIR}")
print(f"📁 Features directory: {FEATURES_DIR}")
print(f"📁 PCA directory: {PCA_DIR}")

📁 Project directory: /app
📁 Data directory: /app/data
📁 Raw data directory: /app/data/raw
📁 Features directory: /app/data/features
📁 PCA directory: /app/data/pca


## Initialize PySpark

### Create SparkSession

In [4]:
spark = (
    SparkSession.builder.appName("P11-Fruits-Local-Development") # Name of the application that will be displayed in the Spark UI
    .master("local[*]") # Local development (all cores) or "yarn" for cluster. Here we use all cores available on the machine. We can replace * by the number of cores we want to use.
    # Memory configuration
    .config("spark.driver.memory", "4g") # Driver memory (4GB)
    .config("spark.executor.memory", "4g") # Executor memory (4GB)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") # Arrow enabled (faster data processing)
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1024") # Max records per batch (1024)
    # After setting all the configurations, we can create or get the SparkSession
    .getOrCreate()
)

# Set log level to WARN
spark.sparkContext.setLogLevel("WARN")

# Get SparkContext for broadcast
sc = spark.sparkContext


print(f"✅ SparkSession created")
print(f"   Version Spark: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   App Name: {spark.sparkContext.appName}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 12:28:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession created
   Version Spark: 3.5.3
   Master: local[*]
   App Name: P11-Fruits-Local-Development


## Data Exploration

### Download data (optional)

In [6]:
import json
from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

kaggle_username = os.getenv("KAGGLE_USERNAME")
kaggle_key = os.getenv("KAGGLE_TOKEN")

assert kaggle_username and kaggle_key, "Variables KAGGLE_USERNAME / KAGGLE_KEY manquantes."

kaggle_dir = os.environ.get("KAGGLE_CONFIG_DIR", "/app/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

print("kaggle.json created in", kaggle_json_path)

!kaggle datasets download -d moltean/fruits -p /app/data/raw --unzip

kaggle.json created in /app/.kaggle/kaggle.json
Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
  3%|█▏                                     | 172M/5.42G [00:05<02:15, 41.6MB/s]^C
  3%|█▏                                     | 177M/5.42G [00:05<02:46, 33.9MB/s]
User cancelled operation


26/03/19 08:55:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


### Explore data


We uploaded the fruit directory into data/raw. We will use fruits-360-original-size for our production service. While developping, we will use fruits-360-100x100

In [5]:
# Get the number of classes in the training set
training_dir = DATA_DIR / "raw" / "fruits" / "fruits-360_dataset" / "fruits-360" / "Training"
test_dir = DATA_DIR / "raw" / "fruits" / "fruits-360_dataset" / "fruits-360" / "Test"

classes = [fruit_class for fruit_class in os.listdir(training_dir) if os.path.isdir(os.path.join(training_dir, fruit_class))]
print(f"Number of training classes: {len(classes)}")
print(f"Example of classes: {classes[:10]}")


# Get the number of images in the training set
num_train_images = sum(len(files) for r, d, files in os.walk(training_dir) if files)
print(f"Number of training images: {num_train_images}")

num_test_images = sum(len(files) for r, d, files in os.walk(test_dir) if files)
print(f"Number of test images: {num_test_images}")

total_images = num_train_images + num_test_images
print(f"Total number of images: {total_images}")


Number of training classes: 131
Example of classes: ['Tomato 4', 'Apple Red Delicious', 'Tomato 3', 'Huckleberry', 'Blueberry', 'Pear Red', 'Banana Lady Finger', 'Melon Piel de Sapo', 'Pear', 'Cherry 1']
Number of training images: 67692
Number of test images: 22688
Total number of images: 90380


### Load dataset with PySpark


In [6]:
# Like the previous notebook, load 300 images
MAX_IMAGES = 300

images = (
        spark.read.format("binaryFile")  # Read the images as binary files
        .option("pathGlobFilter", "*.jpg") # Filter the images to only include .jpg files
        .option("recursiveFileLookup", "true") # Recursively look for files in the directory
        .load(os.path.join(training_dir)) # Load the images from the directory
        .limit(MAX_IMAGES) # Limit the number of images to 300
    )
print(f"Number of images loaded: {images.count()}")


print("Schema of the images DataFrame:")
images.printSchema()

# Let's display the first 5 images
images.show(5)


Number of images loaded: 300
Schema of the images DataFrame:
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)

+--------------------+--------------------+------+--------------------+
|                path|    modificationTime|length|             content|
+--------------------+--------------------+------+--------------------+
|file:/app/data/ra...|2026-03-13 14:07:...|  7437|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7434|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7424|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7423|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7416|[FF D8 FF E0 00 1...|
+--------------------+--------------------+------+--------------------+
only showing top 5 rows



In [7]:
# Add label column with the class name taken from the path
images = images.withColumn("label", element_at(split(images["path"], "/"), -2))

# Display the first 5 images
images.select("path", "label").show(5)

# Display the schema of the images DataFrame
images.printSchema()

# Count the number of images per class
print(f"Classes distribution:")
label_counts = images.groupBy("label").count().orderBy("label")
label_counts.show(20, truncate=False)


+--------------------+--------------+
|                path|         label|
+--------------------+--------------+
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|Pineapple Mini|
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|     Raspberry|
+--------------------+--------------+
only showing top 5 rows

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

Classes distribution:


+--------------+-----+
|label         |count|
+--------------+-----+
|Pineapple Mini|143  |
|Raspberry     |157  |
+--------------+-----+



## Get features with TensorFlow

### Prepare MobileNetV2

The previous worked used a pretrained model, MobileNetV2. The last layer was omitted so we can finetune on our data.

In [8]:
# Load the MobileNetV2 model
print("Loading MobileNetV2 model")
model = MobileNetV2(weights='imagenet',
                    include_top=False, # We don't want the last layer that is the classification layer - Corrected from the original notebook
                    input_shape=(224, 224, 3),
                    pooling="avg")

print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")
print(f"   Features shape: {model.output_shape[1:]}")
# Get the model weights
model_weights = model.get_weights()

print(f"   Tensor weights: {len(model_weights)}")
print(
    f"   Size: {sum([w.nbytes for w in model_weights]) / 1024 / 1024:.2f} MB"
)

model.summary()

Loading MobileNetV2 model
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
   Input shape: (None, 224, 224, 3)
   Output shape: (None, 1280)
   Features shape: (1280,)
   Tensor weights: 260
   Size: 8.61 MB


Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

## Model weight Broadcasting

### Get and broadcast weights

**Context:***
Training a model needs computational ressources. Using Spark is helping to parallelise the training. Data are split into partitions and each partition is sent to the worker by the Driver.

**Problem :** Each executor has to use the model to train its batch. Loading the model everytime is heavy for the ressources and time consuming.

**Solution :** Broadcasting weights.
- Initial load : the model is loaded once by the Driver and broadcast 
  to all executors (avoiding redundant I/O). (Done on the original work)
- After each batch : each executor computes gradients on its partition.
  Gradients are aggregated (averaged) across all workers via AllReduce,
  and the updated weights are broadcast again to keep all executors in sync. (Missing in the original notebook, implemented here)

In [9]:
# Send the model weight to the driver so it can be used by all the workers
broadcast_weights = sc.broadcast(model_weights)

### Use Pandas UDF to use with broadcasted weights

**Problem**: Spark and TensorFlow can't communicate directly.

**Solution**: Create a Pandas UDF to act as a bridge. Each worker receives 
a batch of images as a pd.Series, preprocesses and feeds it to the model, 
then returns the extracted feature vectors back to Spark, 
which stores them in the distributed DataFrame.

In [10]:
@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER) # SCALAR_ITER loads the model once and sends it to each worker
def featurize_udf(content_series_iter):
    local_model = MobileNetV2(weights=None, include_top=False, pooling='avg')
    local_model.set_weights(broadcast_weights.value)

    for content_series in content_series_iter:
        def process(content):
            try:
                img = Image.open(io.BytesIO(content)).convert('RGB').resize((224, 224)) # Convert to RGB protects from some color issues that could happen in production (greyscale, alpha channel, etc.)
                arr = np.expand_dims(img_to_array(img), axis=0)
                return local_model.predict(preprocess_input(arr), verbose=0)[0].tolist()
            except Exception as e:
                return None
        yield content_series.apply(process)

### Extract and save features

In [11]:
# Features extraction
df_features = images.withColumn(
    "features",
    featurize_udf(col("content"))
)

# Filter out images where the extraction failed
df_features = df_features.filter(col("features").isNotNull())

print(f"✅ Features extracted for {df_features.count()} images")
df_features.select("path", "label", "features").show(5, truncate=60)
features_df = images.repartition(20).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

✅ Features extracted for 300 images


+------------------------------------------------------------+--------------+------------------------------------------------------------+
|                                                        path|         label|                                                    features|
+------------------------------------------------------------+--------------+------------------------------------------------------------+
|file:/app/data/raw/fruits/fruits-360_dataset/fruits-360/T...|     Raspberry|[0.1676918, 0.0875426, 0.0, 0.0, 0.43547454, 0.0, 1.50104...|
|file:/app/data/raw/fruits/fruits-360_dataset/fruits-360/T...|     Raspberry|[0.31870013, 0.14381419, 0.0, 0.0, 0.588749, 0.0, 1.13907...|
|file:/app/data/raw/fruits/fruits-360_dataset/fruits-360/T...|Pineapple Mini|[0.0, 5.040998, 0.02691842, 0.0, 0.0, 0.0, 0.13633993, 0....|
|file:/app/data/raw/fruits/fruits-360_dataset/fruits-360/T...|     Raspberry|[0.06850474, 0.008424827, 0.0, 0.0, 0.37396303, 0.0095932...|
|file:/app/data/raw/fruits/

In [ ]:
df_features.cache()
print("✅ Features mises en cache (en mémoire)")
print(f"   {df_features.count()} images en cache")

# OPTION B: Sauvegarder en Parquet (pour datasets >5000 images ou persistance entre sessions)
# Décommenter si nécessaire :
features_output_path = str(FEATURES_DIR / "mobilenetv2_features")
df_features.write.mode("overwrite").parquet(features_output_path)
print(f"✅ Features sauvegardées: {features_output_path}")

In [16]:
# Save features
features_path = FEATURES_DIR / "mobilenetv2_features.parquet"
features_df.write.mode("overwrite").parquet(str(features_path))

print(f"✅ Features saved to {features_path}")

# Load features
features_df = spark.read.parquet(str(features_path))

✅ Features saved to /app/data/features/mobilenetv2_features.parquet


In [21]:
df = pd.read_parquet(features_path, engine='pyarrow')
display(df.head())
print(f"Features have a dimension of {df.loc[0,'features'].shape[0]}")


,path,label,features
0,file:/app/data/raw/fruits/fruits-360_dataset/f...,Pineapple Mini,"[0.0, 4.734359, 0.0, 0.0, 0.0, 0.0, 0.19414088..."
1,file:/app/data/raw/fruits/fruits-360_dataset/f...,Raspberry,"[0.078383796, 0.20401172, 0.0, 0.0, 0.49105906..."
2,file:/app/data/raw/fruits/fruits-360_dataset/f...,Raspberry,"[0.1676918, 0.0875426, 0.0, 0.0, 0.43547454, 0..."
3,file:/app/data/raw/fruits/fruits-360_dataset/f...,Pineapple Mini,"[0.0, 4.6711464, 0.047309447, 0.0, 0.0, 0.0, 0..."
4,file:/app/data/raw/fruits/fruits-360_dataset/f...,Raspberry,"[0.0, 0.9666217, 0.18446872, 0.0, 0.25625953, ..."


Features have a dimension of 1280


## PCA

In [22]:
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, VectorUDT

In [ ]:
# Function to convert array to vector
array_to_vector = udf(lambda a: Vectors.dense(a), VectorUDT())

# Add vector column
df_features_pca = df_features.withColumn("features_vector", array_to_vector("features"))

# Show the first 5 rows
df_features_pca.show(5, truncate=False)

# To avoid recomputing the vector, we can cache the DataFrame
df_features_pca.cache()


